# Stratified sample for manual topic coding


Creates a sample fo 200 reviews for manual coding

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

In [9]:
df = pd.read_csv("data/stylecom_cleaned.csv")
df.head()

,year,season,designer,author,city,date,review
0,2000,Spring,Matt Nye,Armand Limnander,New York,17-Sep-99,Designer Matt Nye's sophomore show featured a ...
1,2000,Spring,Giorgio Armani,Armand Limnander,Milan,29-Sep-99,"Armani proposed a light, feminine silhouette f..."
2,2000,Spring,Eric Bergère,Armand Limnander,Paris,4-Oct-99,Broadway Garnier was the theme for Eric Berg'r...
3,2000,Spring,Céline,Armand Limnander,Paris,7-Oct-99,Getaway glamour was the theme for Celine's str...
4,2000,Spring,Byblos,Armand Limnander,Milan,27-Sep-99,Judo Jetson blends my favorite cartoon charact...


In [10]:
# proportionate stratified sample of 200 reviews by year
N = 200
SEED = 42

year_counts = df["year"].value_counts().sort_index()
stratum_sizes = (year_counts / year_counts.sum() * N).apply(np.floor).astype(int)

# top-up to hit exactly N
shortfall = N - stratum_sizes.sum()
remainders = (year_counts / year_counts.sum() * N) - stratum_sizes
stratum_sizes[remainders.nlargest(shortfall).index] += 1

print(stratum_sizes)
print(f"Total: {stratum_sizes.sum()}")

year
2000     6
2001     6
2002     6
2003     7
2004     7
2005     8
2006    10
2007    12
2008    13
2009    15
2010    17
2011    20
2012    22
2013    24
2014    27
Name: count, dtype: int64
Total: 200


In [11]:
candidates = []
for year, n in stratum_sizes.items():
    pool = df[df["year"] == year]
    candidates.append(pool.sample(n=n, random_state=SEED))

sample = pd.concat(candidates).sort_values(["year", "date"]).reset_index(drop=True)
sample["topic_number"] = ""

print(f"{len(sample)} reviews sampled")
sample.head()

200 reviews sampled


,year,season,designer,author,city,date,review,topic_number
0,2000,Spring,Ellen Tracy,Armand Limnander,New York,14-Sep-99,"There's a fresh, younger look to my collection...",
1,2000,Fall,Tristan Webber,Plum Sykes,London,17-Feb-00,Tristan Webber is London's king of leather a f...,
2,2000,Fall,Cerruti,Hamish Bowles,Paris,2-Mar-00,Peter Speliopoulos takes the Milanese route fo...,
3,2000,Fall,Missoni,Armand Limnander,Milan,22-Feb-00,It's a good fashion moment for Missoni. Prints...,
4,2000,Spring,Prada,Armand Limnander,Milan,27-Sep-99,Miuccia Prada showed today what is certain to ...,


In [ ]:
# attach review_id
noun_tokens_df = pd.read_pickle("checkpoints/df_with_noun_tokens.pkl")

id_cols = ["year", "season", "designer", "author", "city", "date", "review"]
noun_tokens_df = noun_tokens_df.drop_duplicates(subset=id_cols)

sample = sample.merge(
    noun_tokens_df[["doc_id"] + id_cols].rename(columns={"doc_id": "review_id"}),
    on=id_cols, how="left"
)

cols = ["review_id"] + [c for c in sample.columns if c != "review_id"]
sample = sample[cols]

print(sample["review_id"].isna().sum(), "missing review_id")
sample.head()

0 missing review_id


,review_id,year,season,designer,author,city,date,review,topic_number
0,45,2000,Spring,Ellen Tracy,Armand Limnander,New York,14-Sep-99,"There's a fresh, younger look to my collection...",
1,152,2000,Fall,Tristan Webber,Plum Sykes,London,17-Feb-00,Tristan Webber is London's king of leather a f...,
2,144,2000,Fall,Cerruti,Hamish Bowles,Paris,2-Mar-00,Peter Speliopoulos takes the Milanese route fo...,
3,113,2000,Fall,Missoni,Armand Limnander,Milan,22-Feb-00,It's a good fashion moment for Missoni. Prints...,
4,76,2000,Spring,Prada,Armand Limnander,Milan,27-Sep-99,Miuccia Prada showed today what is certain to ...,


In [13]:
sample.to_csv("data/coding_sample_200.csv", index=False)
print("saved")

saved
